In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.read.csv('/FileStore/dados/PORTES_2025.csv', 
                    header=True,
                    inferSchema=True,
                    encoding='latin1',
                    sep=';')

df.display(5)

In [0]:
df.printSchema()


In [0]:
for coluna in df.columns:
    nulos = df.select(F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0))).collect()[0][0]
    print(f"{coluna}: {nulos} nulos")

In [0]:
df.groupBy('MES_MISSAO').count().display()

In [0]:
# df = df.select([
#     F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c)
#     for c, t in df.dtypes
# ])

for coluna in df.columns:
    df = df.withColumn(coluna, F.regexp_replace(coluna,\
            '                                                  ', 'Não Informado'))

df.groupBy('calibre_arma').count().display()

In [0]:
df_sp = df.filter(F.col('UF') == 'SP')
df_sp.display()

In [0]:
df_sp.write.format('delta').mode('overwrite').save('/FileStore/RenanMarques/Job_Databricks_SP')

In [0]:
# df_conf = spark.read.format('delta').load('/FileStore/RenanMarques/Job_Databricks')
# df_conf.display()